# 04 Merge y Filter — entidades de spacyNER y GLiNER

🎯 **Objetivo:** Mezclar las entidades de ambas herramientas y filtrar las que son útiles para el LLM (entidades candidatas para aprobar o descartar).

🔍 **Puntos clave:**
1. Revisar entidades resultantes de un solo un documento (con ambas herramientas)
2. Ajustar parámetros de merge y filter
3. Procesar todos los archivos y exportar candidatos finales a JSON

## 0 · Imports y configuración

In [ ]:
import sys
import json
from pathlib import Path
from collections import defaultdict

import pandas as pd

# Agrega la raíz del proyecto al path para importar src/
PROJECT_ROOT = Path("../").resolve()  # ajusta si se corre desde otro lugar
sys.path.insert(0, str(PROJECT_ROOT))

from src.ner.candidate_merger import CandidateMerger
from src.ner.entity_filter import EntityFilter

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# ── Rutas ──────────────────────────────────────────────────────────────────
SPACY_CANDIDATES   = PROJECT_ROOT / "data" / "processed" / "entidades_candidatas_spacy"
GLINER_CANDIDATES  = PROJECT_ROOT / "data" / "processed" / "entidades_candidatas_gliner"
OUTPUT_DIR         = PROJECT_ROOT / "data" / "processed" / "entidades_candidatas_merged"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Results spaCy:   {SPACY_CANDIDATES}")
print(f"Results GliNER:  {GLINER_CANDIDATES}")
print(f"Salida:          {OUTPUT_DIR}")

## 1 · Revisar solo un documento

In [ ]:
def obtener_predicciones_modelo(directorio: Path, doc_id: str, llave_json: str, min_score: float = 0.0) -> set:
    archivos = list(directorio.glob(f"*{doc_id}*.json"))
    if not archivos:
        return set()
    with open(archivos[0], 'r', encoding='utf-8') as f:
        datos = json.load(f)
        
    lista_candidatos = datos.get(llave_json, [])
    
    # Extraemos el texto, pero APLICAMOS EL FILTRO MATEMÁTICO de score
    predicciones = {
        cand.get("text", "").lower().strip() 
        for cand in lista_candidatos 
        if cand.get("score", 1.0) >= min_score  # Si es spaCy pasa directo, GLiNER se filtra
    }
    return predicciones

In [ ]:
spacy_file = SPACY_CANDIDATES / "C4_transcript_candidates.json"
gliner_file = GLINER_CANDIDATES / "C4_transcript_candidates.json"

In [ ]:
# merger y filter con valores por default
mergerObject = CandidateMerger()
filterObject = EntityFilter()

In [ ]:
merged_doc = mergerObject.merge_from_files(
	spacy_path=spacy_file,
	gliner_path=gliner_file
)

In [ ]:
# devuelve MergedDocument con solo las que pasan, directamente aplicamos los cambios después de revisar
clean_doc = filterObject.filter(merged_doc)   
print(clean_doc.summary())

In [ ]:
# Ver qué se eliminó y por qué
for r in clean_doc.removed:
	print(r["entity"].text, "→", r["reason"])

## 2 · Ajustar parámetros de merge y filter

In [ ]:
# merger y filter ajustando algunos valores
mergerObject = CandidateMerger(gliner_medium_threshold=0.95)
filterObject = EntityFilter(filter_medium=True, extra_blacklist={"temporal", "jornalero"})

In [ ]:
merged_doc = mergerObject.merge_from_files(
	spacy_path=spacy_file,
	gliner_path=gliner_file
)

In [ ]:
filterObject.preview(merged_doc)

## 3 · Procesar todos los archivos y exportar resultados

In [ ]:
#### merger y filter primera prueba
# mergerObject  = CandidateMerger(gliner_medium_threshold=0.75)
# filterObject  = EntityFilter(filter_medium=True)

#### merger y filter al final, porque no se va a leeer todo el documento por el LLM por el costo que implica
mergerObject  = CandidateMerger(gliner_medium_threshold=0.95)
filterObject  = EntityFilter()

In [ ]:
merged_docs = mergerObject.merge_directory(
	spacy_dir=SPACY_CANDIDATES,
	gliner_dir=GLINER_CANDIDATES
)

In [ ]:
print("\nFiltrado y guardado en batch...")
filterObject.filter_batch(merged_docs, OUTPUT_DIR)

In [ ]:
print("\nRevisar uno de los resultados...")
# muestra por entidad y qué se elimina y por qué
filterObject.preview(merged_docs[0])